# Debug: Check Version and Log-Likelihood

In [ ]:
# Check what version is installed
import SearchLibrium
print(f"Installed version: {SearchLibrium.__version__}")

# Check the file location
import inspect
print(f"\nSearchLibrium location: {inspect.getfile(SearchLibrium)}")

In [ ]:
# Make sure it's actually 0.0.99
# If not 0.0.99, uninstall and reinstall
if SearchLibrium.__version__ != "0.0.99":
    print(f"WARNING: Version is {SearchLibrium.__version__}, expected 0.0.99")
    print("\nTo fix:")
    print("1. Run: pip uninstall SearchLibrium -y")
    print("2. Run: pip install SearchLibrium==0.0.99")
    print("3. Restart Jupyter kernel")
else:
    print(f"OK: Version {SearchLibrium.__version__} is correct")

In [ ]:
# Check if Sobol is actually being used
from SearchLibrium.Halton import Draws

draws_check = Draws(k=3, halton_opts=None)
print(f"Sobol is default: {draws_check.halton.use_sobol}")

if not draws_check.halton.use_sobol:
    print("ERROR: Sobol is NOT being used!")
    print("This suggests the old version (0.0.98) is still installed")
else:
    print("OK: Sobol is being used as default")

In [ ]:
# Now run the model with the EXACT same configuration
import numpy as np
import pandas as pd
from SearchLibrium.MixedLogit import MixedLogit

# Load Berlin data
df = pd.read_csv('C:/Users/ahernz/source/SearchLibrium/data/Berlin_Data.csv')
df['PRICE'] = df['PRICE'] * -1

varnames = ['RECRE', 'PRICE', 'CF', 'CF_car', 'CF_stay', 'CF_pt', 'CF_age', 'CF_male',
            'BIKELANE', 'BIKESEP', 'DIST6', 'DIST3', 'FREQ_HIGHER', 'FREQ_HIGHEST',
            'UNGUARDED', 'GUARDED']

randvars = {
    'RECRE': 'n', 'PRICE': 'ln', 'BIKELANE': 'n', 'BIKESEP': 'n',
    'DIST6': 'n', 'DIST3': 'n', 'FREQ_HIGHER': 'n', 'FREQ_HIGHEST': 'n',
    'UNGUARDED': 'n', 'GUARDED': 'n'
}

choice_id = df['csn']
ind_id = df['ID_1']
choice_var = df['Choice_']
alt_var = df['Scenario']

print("Data loaded and configured")

In [ ]:
# Setup model
model = MixedLogit()

print("Setting up model...")
model.setup(
    X=df[varnames],
    y=choice_var,
    varnames=varnames,
    ids=choice_id,
    panels=ind_id,
    alts=alt_var,
    base_alt=None,
    fit_intercept=False,
    n_draws=200,
    randvars=randvars,
    gtol=1e-6,
    ftol=1e-8,
    maxiter=100,
    mnl_init=False  # IMPORTANT: Try with False first
)

print(f"Model setup complete")
print(f"Using Sobol: {model.draws_generator.halton.use_sobol}")

In [ ]:
# CRITICAL: Check INITIAL likelihood WITHOUT fitting
# This is what should be close to -1970.355

print("Checking INITIAL likelihood (without fitting)...\n")

# Generate draws
draws, drawstrans = model.generate_draws(model.N, model.n_draws, halton=True)
model.draws = draws
model.drawstrans = drawstrans

# Create coefficient vector
n_coeff = model.Kf + model.Kr + model.Kchol + model.Kbw + 2*model.Kftrans + 3*model.Krtrans
betas = np.repeat(0.1, n_coeff)

# Compute likelihood at initial point
result = model.get_loglik_gradient(
    betas, model.X, model.y, model.panel_info,
    draws, drawstrans, model.weights, model.avail, model.batch_size
)
ll_initial = result[0]

print(f"INITIAL Log-Likelihood: {ll_initial:.6f}")
print(f"Target (searchlogit): -1970.355")
print(f"Difference: {abs(ll_initial - (-1970.355)):.3f} points")
print(f"\nConfiguration:")
print(f"  - Using Sobol: {model.draws_generator.halton.use_sobol}")
print(f"  - N draws: {model.n_draws}")
print(f"  - Respondents: {model.N}")
print(f"  - Variables: {model.K}")
print(f"  - Random variables: {model.Kr}")

# Check if it's close
if abs(ll_initial - (-1970.355)) < 200:
    print(f"\n[OK] INITIAL likelihood is in acceptable range!")
elif abs(ll_initial - (-1970.355)) < 500:
    print(f"\n[WARN] Gap is moderate, optimization should improve this")
else:
    print(f"\n[ERROR] Gap is too large - check configuration or version")

In [ ]:
# If initial is far off, check what's different
print("\n" + "="*60)
print("DIAGNOSTIC INFORMATION")
print("="*60)

print(f"\nVersion check:")
print(f"  - SearchLibrium.__version__: {SearchLibrium.__version__}")
print(f"  - Should be: 0.0.99")
print(f"  - Match: {SearchLibrium.__version__ == '0.0.99'}")

print(f"\nDraw configuration:")
print(f"  - model.draws_generator.halton.use_sobol: {model.draws_generator.halton.use_sobol}")
print(f"  - Should be: True")

print(f"\nData configuration:")
print(f"  - N respondents: {model.N}")
print(f"  - Expected: 347")
print(f"  - P choices: {model.P}")
print(f"  - J alternatives: {model.J}")
print(f"  - K variables: {model.K}")
print(f"  - Expected K: 16")

print(f"\nLikelihood at initial point:")
print(f"  - Computed LL: {ll_initial:.6f}")
print(f"  - Target: -1970.355")
print(f"  - Gap: {abs(ll_initial - (-1970.355)):.3f}")

In [ ]:
# Now try fitting if initial is good
if abs(ll_initial - (-1970.355)) < 300:
    print("Initial likelihood looks good, proceeding to fit...\n")
    
    model.fit()
    
    print(f"\nFitting complete!")
    print(f"Final LL: {model.loglik:.6f}")
    print(f"Initial LL: {ll_initial:.6f}")
    print(f"Improvement: {ll_initial - model.loglik:.6f}")
    print(f"Gap to target: {abs(model.loglik - (-1970.355)):.3f}")
    print(f"Converged: {model.converged}")
    print(f"Iterations: {model.n_iter}")
else:
    print(f"\nInitial likelihood gap is too large ({abs(ll_initial - (-1970.355)):.1f})")
    print(f"Check:")
    print(f"  1. Version is 0.0.99: {SearchLibrium.__version__}")
    print(f"  2. Sobol is being used: {model.draws_generator.halton.use_sobol}")
    print(f"  3. Data matches expected configuration")